In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import DBSCAN


In [38]:
DATA_PATH = "raw_wholesale_customers.csv"
df = pd.read_csv(DATA_PATH)

In [39]:
print(df.shape)
print("==========Data information===============")
print(df.info())
print(df.head)

(440, 8)
==========Data information===============
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    int64
 7   Delicassen        440 non-null    int64
dtypes: int64(8)
memory usage: 27.6 KB
None
<bound method NDFrame.head of      Channel  Region  Fresh   Milk  Grocery  Frozen  Detergents_Paper  \
0          2       3  12669   9656     7561     214              2674   
1          2       3   7057   9810     9568    1762              3293   
2          2       3   6353   8808     7684    2405              3516   
3          1       3  13265  

In [40]:
# 2 Select Features + IQR Cap
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
X = df[FEATURES].copy()


def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper


low_fresh,  high_fresh = iqr_fun(X["Fresh"])
low_milk,    high_milk = iqr_fun(X["Milk"])
low_grocery, high_grocery = iqr_fun(X["Grocery"])
low_frozen,  high_frozen = iqr_fun(X["Frozen"])
low_det,     high_det = iqr_fun(X["Detergents_Paper"])
low_deli,    high_deli = iqr_fun(X["Delicassen"])


X["Fresh"] = X["Fresh"].clip(lower=low_fresh,  upper=high_fresh)
X["Milk"] = X["Milk"].clip(lower=low_milk,    upper=high_milk)
X["Grocery"] = X["Grocery"].clip(lower=low_grocery, upper=high_grocery)
X["Frozen"] = X["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(
    lower=low_det, upper=high_det)
X["Delicassen"] = X["Delicassen"].clip(lower=low_deli, upper=high_deli)

df[FEATURES] = X



In [41]:
print("\n=== IQR CAP SUMMARY FOR EACH FEATURE ===")
print(
    f"Fresh             low={low_fresh:.2f}  high={high_fresh:.2f}  |  after cap min={X['Fresh'].min():.2f}  max={X['Fresh'].max():.2f}")
print(
    f"Milk              low={low_milk:.2f}  high={high_milk:.2f}  |  after cap min={X['Milk'].min():.2f}  max={X['Milk'].max():.2f}")
print(
    f"Grocery           low={low_grocery:.2f}  high={high_grocery:.2f}  |  after cap min={X['Grocery'].min():.2f}  max={X['Grocery'].max():.2f}")
print(
    f"Frozen            low={low_frozen:.2f}  high={high_frozen:.2f}  |  after cap min={X['Frozen'].min():.2f}  max={X['Frozen'].max():.2f}")
print(
    f"Detergents_Paper  low={low_det:.2f}  high={high_det:.2f}  |  after cap min={X['Detergents_Paper'].min():.2f}  max={X['Detergents_Paper'].max():.2f}")
print(
    f"Delicassen        low={low_deli:.2f}  high={high_deli:.2f}  |  after cap min={X['Delicassen'].min():.2f}  max={X['Delicassen'].max():.2f}")

print("\n=== FEATURES HEAD (after IQR cap) ===")
print(X.head())


=== IQR CAP SUMMARY FOR EACH FEATURE ===
Fresh             low=-17581.25  high=37642.75  |  after cap min=3.00  max=37642.75
Milk              low=-6952.88  high=15676.12  |  after cap min=55.00  max=15676.12
Grocery           low=-10601.12  high=23409.88  |  after cap min=3.00  max=23409.88
Frozen            low=-3475.75  high=7772.25  |  after cap min=25.00  max=7772.25
Detergents_Paper  low=-5241.12  high=9419.88  |  after cap min=3.00  max=9419.88
Delicassen        low=-1709.75  high=3938.25  |  after cap min=3.00  max=3938.25

=== FEATURES HEAD (after IQR cap) ===
     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


In [42]:
# Scale Features
scaler = StandardScaler()
x_scaled = scaler.fit_transform(X)
print("\nScaled shape:", x_scaled.shape)
print(x_scaled)


Scaled shape: (440, 6)
[[ 0.12857261  1.05158597  0.04926747 -0.95324427  0.09579175  0.06589216]
 [-0.42162716  1.08673463  0.35386453 -0.30973493  0.30651872  0.47075856]
 [-0.49064723  0.85804007  0.06793486 -0.04243744  0.38243489  2.46943983]
 ...
 [ 0.31112285  2.38267048  2.45460914 -0.86054235  2.39229863  0.55487464]
 [-0.10466425 -0.70014133 -0.7595007  -0.61070442 -0.75732904  0.7933576 ]
 [-0.84025742 -0.76473271 -0.71730938 -1.01518413 -0.65213577 -1.1228252 ]]


In [43]:
#Elbow Method
print("\n=== ELBOW METHOD (SSE per k) ===")
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(x_scaled)
    print(f"k={k} → SSE={km.inertia_:.2f}")


=== ELBOW METHOD (SSE per k) ===
k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


In [68]:
# 5 Train K-Mean
kmeans = KMeans(n_clusters=5, n_init="auto", random_state=42)
lb = kmeans.fit_predict(x_scaled)

# Add a Cluster column to the dataframe.
df['Cluster'] = lb.astype(int)
print(df.head())

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  DBSCAN  
0     1338.00        0       0  
1     1776.00        0       0  
2     3938.25        0      -1  
3     1788.00        3       0  
4     3938.25        3       2  


In [69]:
#6 Evaluate K-Means
slh = silhouette_score(x_scaled, lb) 
dbi = davies_bouldin_score(x_scaled, lb)
print("\n====================metrics================")
print(f"silhouette_score of Kmeans:     {slh:.3f}")
print(f"davies_bouldin_score of Kmeans: {dbi:.3f}")



====================metrics================
silhouette_score of Kmeans:     0.283
davies_bouldin_score of Kmeans: 1.270


In [52]:
# 7 Research & Train a Second Clustering Algorithm
# DBSCAN
# DBSCAN - Density-Based Spatial Clustering of Applications with Noise. Finds core samples of high density and expands clusters from them.
# This algorithm is particularly good for data which contains clusters of similar density and can find clusters of arbitrary shape.
# Unlike K-means, DBSCAN does not require specifying the number of clusters in advance and can identify outliers as noise points.

# sourc_Link(https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html#dbscan)    


In [62]:
#Train my second Clustering Algorithm
db_scan = DBSCAN(eps=1, min_samples=4)
db_lb = db_scan.fit_predict(x_scaled)
df['DBSCAN'] = db_lb.astype(int)
print(df.head())

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  DBSCAN  
0     1338.00        0       0  
1     1776.00        0       0  
2     3938.25        0      -1  
3     1788.00        3       0  
4     3938.25        3       2  


In [63]:
# Print Silhouette Score for second Clustering Algorithm
slh_dbs =  silhouette_score(x_scaled, db_lb)
dbi_dbscan = davies_bouldin_score(x_scaled, db_lb)

print(f"silhouette_score of DBSCAN:     {slh:.3f}")
print(f"davies_bouldin_score of DBSCAN: {dbi:.3f}")

silhouette_score of DBSCAN:     0.283
davies_bouldin_score of DBSCAN: 1.270


In [54]:
# Print cluster centers in original spend units 
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"

print(centers_df.round(2))


            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         9202.67   6833.30   9104.12  1326.16           3280.12     1871.76
1         8376.23   2150.65   3160.63  1646.33            779.25      674.02
2        17461.54  13805.60  17524.12  4120.57           5460.56     3583.64
3        22346.70   3409.14   3969.33  5819.60            583.07     1566.95
4         4916.98  10768.85  18350.13  1212.37           7780.02      981.37


In [64]:
# 9 Sanity Check
sample_idx = [4, 3, 8]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster", "DBSCAN"]]
print("\n=== SANITY CHECK (3 Clients)===")
print(sanity)



=== SANITY CHECK (3 Clients)===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
8        1       3   5963.0  3648.0   6192.0   425.0            1716.0   

   Delicassen  Cluster  DBSCAN  
4     3938.25        3       2  
3     1788.00        3       0  
8      750.00        1       0  


In [67]:
# Save Output
OUT_PATH = "segmented_wholesale_customers.csv"
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved clustered dataset → {OUT_PATH}")


Saved clustered dataset → segmented_wholesale_customers.csv
